# Module 10: Comprehensions

---

### Learning Objectives
- Write list comprehensions with and without conditions
- Build dict and set comprehensions
- Understand nested comprehensions and generator expressions
- Apply comprehensions to biological data: GC filtering, codon extraction, reverse complement, k-mer analysis

---

## Why comprehensions matter in bioinformatics

Sequence data is almost always processed as collections: all sequences in a FASTA file, all k-mers in a read, all codons in a CDS. Comprehensions replace verbose loops with concise, readable expressions. A codon table built with a dict comprehension, a list of GC-filtered sequences, or a set of unique k-mers are patterns you will write constantly when working with FASTA/FASTQ files, VCF records, and annotation tables.

## How to use this notebook

Run cells from top to bottom on the first pass — later cells depend on functions and variables defined earlier. Once you have run through the notebook, feel free to modify parameters and re-run individual sections.

Each section has runnable examples first, followed by exercises. Try the exercise before looking at the solution cell below it.

## Complicated moments explained

**1. Order of clauses in nested comprehensions**
The loop order in a flat nested comprehension matches nested `for` loops — outer loop first:
```python
# flat: [expr for outer in outer_list for inner in inner_list]
# is the same as:
# for outer in outer_list:
#     for inner in inner_list: result.append(expr)
```
`[[expr for inner in ...] for outer in ...]` gives a list-of-lists instead.

**2. `if` filter vs `if`/`else` transform**
- `if` after `for` is a **filter** (removes items that fail the test).
- `if`/`else` in the **expression** is a **conditional transform** (every item is kept, but mapped differently).
```python
[nt for nt in dna if nt in 'GC']              # filter: keeps only G and C
["purine" if nt in "AG" else "pyrimidine" for nt in dna]  # transform: relabels every base
```

**3. Generator expressions are single-pass**
Once consumed (`sum(...)`, `list(...)`, etc.), the generator is exhausted. If you need to iterate the result more than once, use a list comprehension.

**4. Performance note**
Dict comprehensions with `.count()` inside recompute the count for every unique key. For k-mer counting, `collections.Counter` is faster.

In [ ]:
# Sequence lengths and GC content
sequences = ["ATATATAT", "GCGCGCGC", "ATGCATGC", "AAAGGGCCC", "TTTTAAAA"]

lengths = [len(seq) for seq in sequences]
print(f"Lengths: {lengths}")

gc_values = [
    (seq.count('G') + seq.count('C')) / len(seq) * 100
    for seq in sequences
]

for seq, gc in zip(sequences, gc_values):
    print(f"  {seq:12s}  GC = {gc:.1f}%")

In [ ]:
# Extract codons from a coding sequence
cds = "ATGGCCGATCGATAGCCA"

codons = [cds[i:i+3] for i in range(0, len(cds) - len(cds) % 3, 3)]

print(f"CDS:    {cds}")
print(f"Codons: {codons}")

### 1.2 List Comprehensions with Conditions

In [ ]:
# Filter sequences by GC content
sequences = ["ATATATAT", "GCGCGCGC", "ATGCATGC", "AAAGGGCCC", "TTTTAAAA"]

def gc_content(seq):
    return (seq.count('G') + seq.count('C')) / len(seq) * 100

gc_rich = [seq for seq in sequences if gc_content(seq) > 50]

print("GC-rich sequences (>50%):")
for seq in gc_rich:
    print(f"  {seq}: {gc_content(seq):.1f}%")

In [ ]:
# Find all positions of a motif
dna = "ATGATGCCCATGATGATG"
motif = "ATG"

positions = [i for i in range(len(dna) - len(motif) + 1) if dna[i:i+len(motif)] == motif]

print(f"Sequence: {dna}")
print(f"'{motif}' found at positions: {positions}")

In [ ]:
# if-else in the expression -- conditional transformation
# Classify nucleotides as purine (A, G) or pyrimidine (C, T)
dna = "ATGCGATCG"

classification = [
    "purine" if nt in "AG" else "pyrimidine"
    for nt in dna
]

for nt, cls in zip(dna, classification):
    print(f"  {nt} -> {cls}")

In [ ]:
# Filter AND transform: keep only long sequences, return (name, gc%)
fasta_data = {
    "seq1": "ATG",
    "seq2": "GCGCGCGCGCGC",
    "seq3": "ATATATATATATAT",
    "seq4": "ATGCATGCATGCATGC",
    "seq5": "GC",
}

results = [
    (name, gc_content(seq))
    for name, seq in fasta_data.items()
    if len(seq) > 10
]

print("Long sequences and their GC%:")
for name, gc in results:
    print(f"  {name}: {gc:.1f}%")

---

## 2. Dictionary Comprehensions

```
+----------------------------------------------------------------+
|              DICT COMPREHENSION ANATOMY                         |
+----------------------------------------------------------------+
|                                                                 |
|   {key_expr: val_expr  for item in iterable  if condition}      |
|                                                                 |
|   Example:                                                      |
|   {seq: len(seq) for seq in sequences}                          |
|   -> {'ATGC': 4, 'GGCAT': 5, ...}                              |
|                                                                 |
+----------------------------------------------------------------+
```

In [ ]:
# Count nucleotides in a sequence
dna = "ATGCGATCGATCGTAGCGATCGATCG"

nt_counts = {nt: dna.count(nt) for nt in 'ATGC'}
print(f"Nucleotide counts: {nt_counts}")

# Create a complement mapping and its reverse
complement = {b: c for b, c in [('A','T'), ('T','A'), ('G','C'), ('C','G')]}
reverse_map = {v: k for k, v in complement.items()}
print(f"Complement: {complement}")
print(f"Reverse:    {reverse_map}")

In [ ]:
# Map sequence IDs to GC content
fasta = {
    "gene_A": "ATGCGATCGATCG",
    "gene_B": "GCGCGCGCGCGC",
    "gene_C": "ATATATATATATAT",
    "gene_D": "ATGCATGCATGC",
}

gc_map = {name: gc_content(seq) for name, seq in fasta.items()}

print("GC content per sequence:")
for name, gc in gc_map.items():
    print(f"  {name}: {gc:.1f}%")

In [ ]:
# K-mer counting with dict comprehension
dna = "ATGATGATGATG"
k = 3

kmers = [dna[i:i+k] for i in range(len(dna) - k + 1)]
kmer_counts = {kmer: kmers.count(kmer) for kmer in set(kmers)}

print(f"Sequence: {dna}")
print(f"{k}-mer counts:")
for kmer, count in sorted(kmer_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {kmer}: {count}")

In [ ]:
# Filter a dictionary -- keep only long genes
gene_lengths = {
    "BRCA1": 81189, "TP53": 19149, "EGFR": 188307, "MYC": 4150, "KRAS": 45693,
}

long_genes = {gene: bp for gene, bp in gene_lengths.items() if bp > 50000}
print(f"Genes longer than 50 kb: {long_genes}")

---

## 3. Set Comprehensions

```
+----------------------------------------------------------------+
|               SET COMPREHENSION ANATOMY                         |
+----------------------------------------------------------------+
|                                                                 |
|   {expression  for item in iterable  if condition}              |
|                                                                 |
|   Same as list comprehension but with {} instead of []          |
|   Result: unique values only, unordered                         |
|                                                                 |
+----------------------------------------------------------------+
```

In [ ]:
# Unique k-mers in a sequence
dna = "ATGATGATGATGCGATCG"
k = 3

unique_kmers = {dna[i:i+k] for i in range(len(dna) - k + 1)}

print(f"Sequence: {dna}")
print(f"Unique {k}-mers ({len(unique_kmers)}): {sorted(unique_kmers)}")

In [ ]:
# Unique amino acids in a protein
protein = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
unique_aa = {aa for aa in protein}
print(f"Unique amino acids ({len(unique_aa)}): {sorted(unique_aa)}")

# Codons that encode a specific amino acid
GENETIC_CODE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}

leu_codons = {codon for codon, aa in GENETIC_CODE.items() if aa == 'L'}
ser_codons = {codon for codon, aa in GENETIC_CODE.items() if aa == 'S'}
print(f"\nLeucine codons: {sorted(leu_codons)}")
print(f"Serine codons:  {sorted(ser_codons)}")

---

## 4. Nested Comprehensions

```
+----------------------------------------------------------------+
|                NESTED COMPREHENSION                              |
+----------------------------------------------------------------+
|                                                                  |
|   Flattened (single list):                                       |
|   [expr for outer in outer_list for inner in inner_list]         |
|                                                                  |
|   Nested (list of lists):                                        |
|   [[expr for inner in inner_list] for outer in outer_list]       |
|                                                                  |
|   Note: the outer loop comes FIRST in flattened form             |
|                                                                  |
+----------------------------------------------------------------+
```

In [ ]:
# Generate all 16 dinucleotides and all 64 codons
bases = 'ATGC'

dinucleotides = [b1 + b2 for b1 in bases for b2 in bases]
print(f"All {len(dinucleotides)} dinucleotides: {dinucleotides}")

all_codons = [b1 + b2 + b3 for b1 in bases for b2 in bases for b3 in bases]
print(f"\nTotal codons: {len(all_codons)}")
print(f"First 8: {all_codons[:8]}")
print(f"Last 8:  {all_codons[-8:]}")

In [ ]:
# Three reading frames of a DNA sequence (nested -- list of lists)
dna = "ATGCGATCGATCGTAGCGATCGATCG"

reading_frames = [
    [dna[i:i+3] for i in range(frame, len(dna) - 2, 3)]
    for frame in range(3)
]

print(f"DNA: {dna}")
for idx, codons in enumerate(reading_frames):
    print(f"  Frame +{idx + 1}: {codons}")

# Flatten: all codons from all frames
all_frame_codons = [codon for frame in reading_frames for codon in frame]
print(f"\nAll codons combined: {all_frame_codons}")

---

## 5. Generator Expressions

A generator expression looks like a list comprehension but uses `()` instead of `[]`. It produces items **lazily** (one at a time), saving memory when working with large datasets.

```
+----------------------------------------------------------------+
|           GENERATOR vs LIST COMPREHENSION                       |
+----------------------------------------------------------------+
|                                                                 |
|   List:      [x**2 for x in range(1000000)]                    |
|              -> Creates 1,000,000 items in memory NOW           |
|                                                                 |
|   Generator: (x**2 for x in range(1000000))                    |
|              -> Creates items ONE AT A TIME when needed         |
|                                                                 |
+----------------------------------------------------------------+
```

In [ ]:
import sys

n = 100_000

list_comp = [x ** 2 for x in range(n)]
gen_expr  = (x ** 2 for x in range(n))

print(f"List size:      {sys.getsizeof(list_comp):>10,} bytes")
print(f"Generator size: {sys.getsizeof(gen_expr):>10,} bytes")
print(f"Memory ratio:   {sys.getsizeof(list_comp) / sys.getsizeof(gen_expr):.0f}x")

In [ ]:
# Generators work seamlessly with built-in functions
sequences = ["ATGCGATCG", "GCGCGCGCGC", "ATATATATAT", "ATGCATGCAT"]

avg_gc = sum(gc_content(s) for s in sequences) / len(sequences)
max_len = max(len(s) for s in sequences)
any_pure_gc = any(set(s) <= {'G', 'C'} for s in sequences)
all_valid = all(set(s) <= {'A', 'T', 'G', 'C'} for s in sequences)

print(f"Average GC: {avg_gc:.1f}%")
print(f"Max length: {max_len} bp")
print(f"Any pure GC: {any_pure_gc}")
print(f"All valid DNA: {all_valid}")

In [ ]:
# Memory-efficient processing of a large dataset
import random
random.seed(42)

large_dataset = [
    ''.join(random.choices('ATGC', k=random.randint(50, 500)))
    for _ in range(10_000)
]

# Count sequences with GC > 55% using a generator (no intermediate list)
high_gc_count = sum(1 for seq in large_dataset if gc_content(seq) > 55)

print(f"Total sequences: {len(large_dataset):,}")
print(f"Sequences with GC > 55%: {high_gc_count:,}")
print(f"Fraction: {high_gc_count / len(large_dataset):.1%}")

---

## 6. Bioinformatics Applications

### 6.1 Reverse Complement in One Line

In [ ]:
complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
dna = "ATGCGATCGATCG"

# Reverse complement in one expression
rev_comp = ''.join(complement[nt] for nt in reversed(dna))

print(f"Original:    5'-{dna}-3'")
print(f"Rev. comp.:  3'-{rev_comp}-5'")

### 6.2 Translation Using Comprehensions

In [ ]:
# Translate DNA to protein -- comprehension-style
cds = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGA"

codons = [cds[i:i+3] for i in range(0, len(cds) - len(cds) % 3, 3)]
protein_full = [GENETIC_CODE.get(c, 'X') for c in codons]

# Take only up to the first stop codon
protein = []
for aa in protein_full:
    if aa == '*':
        break
    protein.append(aa)
protein = ''.join(protein)

print(f"DNA:     {cds}")
print(f"Codons:  {codons}")
print(f"Protein: {protein}")

### 6.3 Filter Sequences by Multiple Criteria

In [ ]:
sequences = {
    "seq1": "ATGCGATCGATCG",
    "seq2": "GCGCGCGCGCGC",
    "seq3": "ATATAT",
    "seq4": "ATGCATGCATGCATGC",
    "seq5": "GGGGCCCC",
}

# Dict comprehension: keep sequences with length > 10 AND GC between 40-60%
filtered = {
    name: seq
    for name, seq in sequences.items()
    if len(seq) > 10 and 40 <= gc_content(seq) <= 60
}

print("Filtered (len>10, GC 40-60%):")
for name, seq in filtered.items():
    print(f"  {name}: {seq} (len={len(seq)}, GC={gc_content(seq):.1f}%)")

### 6.4 Open Reading Frames and Codon Degeneracy

In [ ]:
# Find all start and stop codon positions
dna = "ATGAAATTTCCCATGGGGTAGAAATGCCCCCCTGA"

starts = [i for i in range(len(dna) - 2) if dna[i:i+3] == 'ATG']
stops  = [i for i in range(len(dna) - 2) if dna[i:i+3] in ('TAA', 'TAG', 'TGA')]

print(f"DNA: {dna}")
print(f"Start codon (ATG) positions: {starts}")
print(f"Stop codon positions:        {stops}")

# Find ORFs: pair each start with the nearest downstream in-frame stop
orfs = []
for start in starts:
    for stop in stops:
        if stop > start and (stop - start) % 3 == 0:
            orfs.append((start, stop + 3, dna[start:stop + 3]))
            break

print("\nOpen Reading Frames:")
for s, e, seq in orfs:
    print(f"  {s}-{e}: {seq} ({len(seq)} bp)")

In [ ]:
# Codon degeneracy: map amino acid to number of synonymous codons
from collections import Counter

codon_counts_per_aa = Counter(GENETIC_CODE.values())

degeneracy = {
    aa: count
    for aa, count in sorted(codon_counts_per_aa.items(), key=lambda x: x[1], reverse=True)
}

print(f"{'AA':<4} {'Codons':>6}  Degeneracy")
print("-" * 30)
for aa, count in degeneracy.items():
    bar = '#' * count
    label = "(stop)" if aa == '*' else ""
    print(f"{aa:<4} {count:>6}  {bar} {label}")

---

## 7. Exercises

### Exercise 1: K-mer Spectrum (One-Liner)

Write a **single expression** that returns a sorted list of all unique k-mers in a DNA sequence.

In [ ]:
# Exercise 1: K-mer spectrum

dna = "ATGATGATGCGATCG"
k = 3

# Write a single expression that produces a sorted list of unique k-mers
# YOUR CODE HERE
# spectrum = ...

In [ ]:
# --- Solution ---

dna = "ATGATGATGCGATCG"
k = 3

spectrum = sorted({dna[i:i+k] for i in range(len(dna) - k + 1)})

print(f"Sequence: {dna}")
print(f"Unique {k}-mers (sorted): {spectrum}")

### Exercise 2: Codon Frequency Table (One-Liner)

Write a dict comprehension that maps each codon in a CDS to its count.

In [ ]:
# Exercise 2: Codon frequency as a dict comprehension

cds = "ATGATGGCCGCCATGGCCGATCGA"

# Write a dict comprehension that maps codon -> count
# Hint: first extract codons, then use a comprehension
# YOUR CODE HERE
# codon_freq = ...

In [ ]:
# --- Solution ---

cds = "ATGATGGCCGCCATGGCCGATCGA"

codons = [cds[i:i+3] for i in range(0, len(cds) - len(cds) % 3, 3)]
codon_freq = {c: codons.count(c) for c in set(codons)}

print(f"CDS: {cds}")
print(f"Codons: {codons}")
print(f"Frequency: {codon_freq}")

### Exercise 3: Reverse Complement (One-Liner Challenge)

Write the reverse complement of a DNA string as a single expression, **without** defining a helper dictionary beforehand.

In [ ]:
# Exercise 3: Reverse complement one-liner

dna = "ATGCGATCGATCG"

# Write a single expression that produces the reverse complement
# YOUR CODE HERE
# rev_comp = ...

In [ ]:
# --- Solution ---

dna = "ATGCGATCGATCG"

rev_comp = ''.join({'A':'T','T':'A','G':'C','C':'G'}[nt] for nt in dna[::-1])

print(f"DNA:      5'-{dna}-3'")
print(f"Rev comp: 3'-{rev_comp}-5'")

### Exercise 4: Amino Acid Composition Pipeline

Given a DNA sequence, write a pipeline of comprehensions that:
1. Extracts codons
2. Translates to amino acids
3. Counts the frequency of each amino acid

Use comprehensions only (no explicit loops).

In [ ]:
# Exercise 4: AA composition pipeline

dna = "ATGGCCGATCGAGCCGCCATGGCCGATCGA"

# Step 1: extract codons (list comprehension)
# Step 2: translate each codon (list comprehension)
# Step 3: count each amino acid (dict comprehension)
# YOUR CODE HERE

In [ ]:
# --- Solution ---

dna = "ATGGCCGATCGAGCCGCCATGGCCGATCGA"

codons = [dna[i:i+3] for i in range(0, len(dna) - len(dna) % 3, 3)]
amino_acids = [GENETIC_CODE.get(c, 'X') for c in codons]
composition = {aa: amino_acids.count(aa) for aa in set(amino_acids)}

print(f"DNA:     {dna}")
print(f"Codons:  {codons}")
print(f"AAs:     {''.join(amino_acids)}")
print(f"\nComposition:")
for aa, count in sorted(composition.items(), key=lambda x: x[1], reverse=True):
    print(f"  {aa}: {count}  {'#' * (count * 3)}")

### Exercise 5: One-Liner Challenges

Solve each of these in a single comprehension expression.

In [ ]:
# Challenge A: RNA transcription -- replace T with U
dna = "ATGCGATCGATCG"
# YOUR CODE HERE
# rna = ...

# Challenge B: Extract codons that are NOT stop codons
cds = "ATGGCCGATCGATAGCCAATGGATCGATAGCCGATCGATAGCCA"
# YOUR CODE HERE
# non_stop_codons = ...

# Challenge C: Create a dict mapping gene name to True/False (is GC-rich >50%?)
genes = {"gA": "GCGCGCGC", "gB": "ATATATATAT", "gC": "ATGCATGC"}
# YOUR CODE HERE
# gc_rich_map = ...

In [ ]:
# --- Solutions ---

# Challenge A: RNA transcription
dna = "ATGCGATCGATCG"
rna = ''.join('U' if nt == 'T' else nt for nt in dna)
print(f"DNA: {dna}")
print(f"RNA: {rna}")

# Challenge B: Non-stop codons
cds = "ATGGCCGATCGATAGCCAATGGATCGATAGCCGATCGATAGCCA"
stop_codons = {'TAA', 'TAG', 'TGA'}
non_stop_codons = [
    cds[i:i+3]
    for i in range(0, len(cds) - len(cds) % 3, 3)
    if cds[i:i+3] not in stop_codons
]
print(f"\nNon-stop codons: {non_stop_codons}")

# Challenge C: GC-rich map
genes = {"gA": "GCGCGCGC", "gB": "ATATATATAT", "gC": "ATGCATGC"}
gc_rich_map = {name: gc_content(seq) > 50 for name, seq in genes.items()}
print(f"\nGC-rich map: {gc_rich_map}")

---

## Key Takeaways

```
+----------------------------------------------------------------+
|                  COMPREHENSION SUMMARY                          |
+----------------------------------------------------------------+
|                                                                 |
|  List:       [expr for x in iterable]         -> [1, 2, 3]     |
|  Dict:       {k: v for x in iterable}         -> {a: 1, b: 2}  |
|  Set:        {expr for x in iterable}         -> {1, 2, 3}     |
|  Generator:  (expr for x in iterable)         -> lazy iterator |
|                                                                 |
|  Bioinformatics patterns:                                       |
|    K-mer extraction:  [seq[i:i+k] for i in range(...)]         |
|    Codon splitting:   [seq[i:i+3] for i in range(0, len, 3)]   |
|    Sequence filter:   [s for s in seqs if gc(s) > 50]          |
|    Translation:       [code[c] for c in codons]                |
|    Rev complement:    ''.join(comp[n] for n in seq[::-1])      |
|                                                                 |
|  Guidelines:                                                    |
|    - Use comprehensions for simple transformations              |
|    - Use generators for large datasets (memory efficient)       |
|    - Keep comprehensions readable -- if it needs 3+ lines,      |
|      consider a regular loop instead                            |
|                                                                 |
+----------------------------------------------------------------+
```

---

### Next: Module 11 -- Iterators and Generators

---

[< Previous: Dictionaries and Sets](../09_Dictionaries_and_Sets/02_sets.ipynb) | [Tier 1: Python for Bioinformatics Overview](../README.md) | [Next: Iterators >](../11_Iterators_and_Generators/01_iterators.ipynb)

---